# Módulo AI7 · Inferencia y despliegue — Práctica (CPU)

Construimos, con un Transformer de juguete, **las piezas que hay dentro de un servidor de inferencia** y medimos cada decisión:

1. Entrenamos un modelo diminuto con una tarea comprobable.
2. **KV-cache** implementada de verdad: correcta (misma salida) y medida (más rápida).
3. **Batching**, **TTFT** frente a tokens/s, y **continuous batching** simulado con una cola.
4. **Cuantización** int8 e int4 a mano, con error numérico *y* calidad del modelo.
5. **Memoria de KV-cache** de un modelo de 7–8 B con contexto largo.
6. **Coste por millón de tokens** propio frente a API.
7. **Prueba de carga** contra una API FastAPI local (sin servidores colgados) y esqueleto de **vLLM en Colab**.

Todo corre en CPU en pocos minutos y sin clave de API. Los tiempos absolutos dependerán de tu máquina: lo que importa son las **formas** de las curvas.

## 0 · Preparación: modelo con KV-cache y una tarea comprobable

El modelo es el del juguete del AI5, ampliado para aceptar una **caché** de claves y valores. Lo entrenamos 400 pasos en una tarea sencilla (el siguiente token sigue una regla, con un 10 % de ruido) para poder medir la **calidad** más adelante, no solo el tamaño.

In [ ]:
import asyncio
import copy
import math
import statistics
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

def mejor_de(fn, repeticiones=3):
    """Ejecuta fn varias veces y devuelve el MEJOR tiempo (el más limpio de ruido del sistema)."""
    mejor = float("inf")
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        fn()
        mejor = min(mejor, time.perf_counter() - t0)
    return mejor

In [ ]:
class Bloque(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.qkv, self.proj = nn.Linear(dim, 3 * dim), nn.Linear(dim, dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.h = heads

    def forward(self, x, cache=None):
        B, T, C = x.shape                       # T = tokens NUEVOS que entran ahora
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, -1).transpose(1, 2) for t in (q, k, v))
        if cache is not None:                   # KV-cache: añadimos K y V nuevas a las guardadas
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        S = k.size(2)                           # longitud total = prefijo en caché + tokens nuevos
        a = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))          # (B, h, T, S)
        visible = torch.ones(T, S, dtype=torch.bool).tril(diagonal=S - T)   # máscara causal
        a = a.masked_fill(~visible, float("-inf"))
        a = (F.softmax(a, -1) @ v).transpose(1, 2).reshape(B, T, C)
        return x + self.mlp(self.ln2(x + self.proj(self.ln1(a))))

class Modelo(nn.Module):
    """Transformer decoder mínimo (misma idea que el juguete del AI5), con soporte de KV-cache."""
    def __init__(self, vocab=64, dim=64, capas=2, heads=4, max_len=64):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, dim), nn.Embedding(max_len, dim)
        self.blocks = nn.ModuleList([Bloque(dim, heads) for _ in range(capas)])
        self.head = nn.Linear(dim, vocab)

    def forward(self, idx, caches=None, pos0=0):
        x = self.tok(idx) + self.pos(torch.arange(pos0, pos0 + idx.shape[1]))
        for i, bloque in enumerate(self.blocks):
            x = bloque(x, None if caches is None else caches[i])
        return self.head(x)

V = 64   # vocabulario de la tarea de juguete

def lote(n, T, g):
    """Secuencias donde el siguiente token es (7·token + 3) mod 64 el 90 % de las veces."""
    x = torch.zeros(n, T, dtype=torch.long)
    x[:, 0] = torch.randint(0, V, (n,), generator=g)
    for t in range(1, T):
        regla = (7 * x[:, t - 1] + 3) % V
        azar = torch.randint(0, V, (n,), generator=g)
        x[:, t] = torch.where(torch.rand(n, generator=g) < 0.9, regla, azar)
    return x

g = torch.Generator().manual_seed(0)
m = Modelo(vocab=V, dim=64, capas=2, heads=4, max_len=64)
opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
t0 = time.perf_counter()
for paso in range(400):
    x = lote(64, 32, g)
    perdida = F.cross_entropy(m(x[:, :-1]).reshape(-1, V), x[:, 1:].reshape(-1))
    opt.zero_grad(); perdida.backward(); opt.step()
m.eval()

X_EVAL = lote(500, 32, torch.Generator().manual_seed(123))     # conjunto de evaluación fijo
@torch.no_grad()
def evaluar(modelo):
    """Calidad = % de veces que acierta la regla de la tarea; y KL frente al modelo fp32 original."""
    logits = modelo(X_EVAL[:, :-1])
    acierto = (logits.argmax(-1) == (7 * X_EVAL[:, :-1] + 3) % V).float().mean().item()
    return acierto, logits

ACIERTO_FP32, LOGITS_FP32 = evaluar(m)
print(f"Modelo entrenado en {time.perf_counter() - t0:.1f} s · pérdida final {perdida.item():.3f}")
print(f"Acierto de la regla (fp32): {ACIERTO_FP32:.1%}   (techo ≈ 100 %: el 10 % de ruido no se predice, la regla sí)")

## 1 · KV-cache: cuánto cuesta cada paso

Al generar token a token, el paso *t* necesita atender a los *t* tokens anteriores. **Sin caché** se recalculan las claves y valores (K, V) de todo el prefijo en cada paso: se procesan *t* tokens y cada uno mira hasta *t* claves, así que el paso cuesta **O(t²)** y generar *T* tokens cuesta **O(T³)** en atención. **Con caché** solo se procesa el token nuevo, que mira *t* claves: **O(t)** por paso y **O(T²)** en total. (Además, sin caché las capas lineales y el MLP también reprocesan todo el prefijo: la caché ahorra trabajo en todo el modelo, no solo en la atención.)

In [ ]:
def pares_sin_cache(T):
    """Sin caché, el paso t reprocesa el prefijo entero: t consultas, cada una mira hasta t claves."""
    return sum(t * (t + 1) // 2 for t in range(1, T + 1))     # O(t²) por paso → O(T³) en total

def pares_con_cache(T):
    """Con caché, el paso t procesa SOLO el token nuevo: 1 consulta que mira t claves."""
    return sum(t for t in range(1, T + 1))                    # O(t) por paso → O(T²) en total

print(f"{'T':>5} {'sin caché':>14} {'con caché':>12} {'ratio':>8}   (pares consulta-clave en la atención)")
for T in (32, 64, 128, 512):
    a, b = pares_sin_cache(T), pares_con_cache(T)
    print(f"{T:>5} {a:>14,} {b:>12,} {a / b:>7.1f}×")
# Fórmulas cerradas: T(T+1)(T+2)/6 frente a T(T+1)/2  →  el ratio tiende a (T+2)/3: CRECE con T.
assert pares_sin_cache(32) == 32 * 33 * 34 // 6 and pares_con_cache(32) == 32 * 33 // 2

**Lectura:** el ratio crece con *T* (≈ (T+2)/3), no se queda en 11×: cuanto más largo el contexto, más gana la caché. Pasamos de la cuenta teórica a la **implementación real** en el modelo de juguete.

In [ ]:
@torch.no_grad()
def generar(modelo, prompt, n_nuevos, usar_cache, tiempos=None):
    """Generación voraz (greedy). Si se pasa `tiempos`, guarda lo que tarda cada paso."""
    idx = prompt.clone()
    caches = [dict() for _ in modelo.blocks] if usar_cache else None
    logits = modelo(idx, caches, 0)                    # prefill: se procesa el prompt entero una vez
    for _ in range(n_nuevos):
        t0 = time.perf_counter()
        siguiente = logits[:, -1].argmax(-1, keepdim=True)
        idx = torch.cat([idx, siguiente], dim=1)
        if usar_cache:
            logits = modelo(siguiente, caches, idx.shape[1] - 1)    # solo el token nuevo
        else:
            logits = modelo(idx)                                    # todo el prefijo otra vez
        if tiempos is not None:
            tiempos.append(time.perf_counter() - t0)
    return idx

# 1) Correctitud: con y sin caché el resultado debe ser IDÉNTICO (si no, la caché está mal hecha)
prompt = lote(4, 8, torch.Generator().manual_seed(7))
a = generar(m, prompt, 40, usar_cache=False)
b = generar(m, prompt, 40, usar_cache=True)
print("Misma salida con y sin KV-cache:", torch.equal(a, b))

# 2) Velocidad: modelo MÁS GRANDE para que el cálculo domine sobre la sobrecarga de Python
grande = Modelo(vocab=V, dim=128, capas=4, heads=4, max_len=512).eval()
prompt = torch.randint(0, V, (1, 8))
print(f"\n{'tokens nuevos':>14} {'sin caché':>11} {'con caché':>11} {'aceleración':>12}")
for n in (32, 64, 128, 256):
    t_sin = mejor_de(lambda: generar(grande, prompt, n, False), 2)
    t_con = mejor_de(lambda: generar(grande, prompt, n, True), 2)
    print(f"{n:>14} {t_sin * 1000:>8.0f} ms {t_con * 1000:>8.0f} ms {t_sin / t_con:>10.1f}×")

# 3) La forma de la curva: tiempo de un paso al principio y al final de la generación
ts_sin, ts_con = [], []
generar(grande, prompt, 256, False, ts_sin)
generar(grande, prompt, 256, True, ts_con)
for nombre, ts in (("sin caché", ts_sin), ("con caché", ts_con)):
    print(f"{nombre}: paso 10 = {np.mean(ts[5:15]) * 1000:.2f} ms · paso 250 = {np.mean(ts[245:255]) * 1000:.2f} ms")

**Lectura:** primero, la caché es *correcta* (misma salida). Segundo, sin caché el tiempo de cada paso **crece** con la longitud; con caché se mantiene casi plano (crece muy poco, porque la atención sigue mirando un prefijo cada vez mayor). El precio: **memoria**. La caché hay que guardarla para cada petición activa (sección 5).

## 2 · Batching, TTFT y tokens/s

Leer los pesos del modelo es lo caro en cada paso de decodificación; si una misma lectura sirve a *B* peticiones, el **throughput total** sube. Medimos un paso de decodificación (con KV-cache) variando el tamaño del lote:

In [ ]:
@torch.no_grad()
def paso_decode(B, longitud_contexto=32, pasos=12):
    """Tiempo medio de UN paso de decodificación con B peticiones a la vez (con KV-cache)."""
    caches = [dict() for _ in grande.blocks]
    grande(torch.randint(0, V, (B, longitud_contexto)), caches, 0)              # prefill
    siguiente = torch.randint(0, V, (B, 1))
    t0 = time.perf_counter()
    for i in range(pasos):
        grande(siguiente, caches, longitud_contexto + i)
    return (time.perf_counter() - t0) / pasos

TAM_LOTE = (1, 2, 4, 8, 16, 32, 64)
paso_decode(1)                                                                  # calentamiento
tiempos_paso = {B: min(paso_decode(B) for _ in range(3)) for B in TAM_LOTE}
print(f"{'lote B':>7} {'ms/paso':>9} {'tok/s TOTAL':>12} {'tok/s por usuario':>18}")
for B, t in tiempos_paso.items():
    print(f"{B:>7} {t * 1000:>9.2f} {B / t:>12,.0f} {1 / t:>18,.0f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.2))
Bs = list(tiempos_paso)
ax.plot(Bs, [B / t for B, t in tiempos_paso.items()], "o-", label="tokens/s totales (throughput)")
ax.plot(Bs, [1 / t for t in tiempos_paso.values()], "s--", label="tokens/s que ve CADA usuario")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("tamaño del lote"); ax.set_ylabel("tokens por segundo")
ax.set_title("Batching: el servidor gana, cada usuario pierde un poco")
ax.legend(); fig.tight_layout(); plt.show()

**Lectura:** el total sube con *B*, pero los tokens/s que ve **cada usuario** bajan. En una GPU real, con un modelo grande, el paso apenas tarda más con lotes moderados (porque lo limita la lectura de pesos, no el cálculo), así que la ganancia es mayor que en esta CPU; la forma de las dos curvas es la misma.

In [ ]:
@torch.no_grad()
def ttft(L):
    """Tiempo hasta el primer token = prefill del prompt + elegir el primer token."""
    prompt = torch.randint(0, V, (1, L))
    def una_vez():
        caches = [dict() for _ in grande.blocks]
        grande(prompt, caches, 0)[:, -1].argmax(-1)
    return mejor_de(una_vez, 5)

ttft(16)                                                             # calentamiento
vel_decode = 1 / tiempos_paso[1]
print(f"Velocidad de generación (B=1): {vel_decode:,.0f} tokens/s  → cada token tarda {1000 / vel_decode:.2f} ms")
print(f"\n{'prompt (tokens)':>16} {'TTFT':>10}")
for L in (16, 64, 256, 500):
    print(f"{L:>16} {ttft(L) * 1000:>7.1f} ms")
print("\nDos métricas distintas: el TTFT lo manda el PREFILL (depende del prompt); los tokens/s los manda el DECODE.")
print("Un chat se siente rápido si el TTFT es bajo (streaming); una tarea por lotes solo mira el throughput.")

## 3 · *Continuous batching* con una cola

En el *batching* estático, el servidor forma un lote y **no admite a nadie más hasta que acaba la petición más larga**; los huecos de las que terminan antes se quedan vacíos. El *continuous batching* (lo que hace vLLM) rellena cada hueco libre en el paso siguiente. Lo simulamos con 80 peticiones de longitudes muy desiguales y 8 huecos:

In [ ]:
rng = np.random.default_rng(0)
N_PET = 80
llegadas = np.cumsum(rng.exponential(1 / 0.45, N_PET)).astype(int)         # ~0,45 peticiones por paso
salidas = np.clip(rng.lognormal(3.2, 0.8, N_PET).astype(int), 5, 150)      # longitudes MUY desiguales
PETICIONES = list(zip(llegadas.tolist(), salidas.tolist()))
print(f"{N_PET} peticiones · longitud de salida: mediana {int(np.median(salidas))}, "
      f"p90 {int(np.percentile(salidas, 90))}, máx {salidas.max()} tokens")

def simular(politica, peticiones, slots=8):
    """Tiempo discreto: 1 unidad = 1 paso de decodificación del lote (coste constante con ≤ slots).
    politica='estatico': espera a que acabe TODO el lote antes de admitir peticiones nuevas.
    politica='continuo': cada hueco libre se rellena en el siguiente paso (continuous batching)."""
    cola = sorted(range(len(peticiones)), key=lambda i: peticiones[i][0])
    pendientes = list(cola)
    activas = {}                                   # id → tokens que le faltan
    inicio, fin, ocupados = {}, {}, 0
    t = 0
    while pendientes or activas:
        puede_admitir = (politica == "continuo") or (politica == "estatico" and not activas)
        if puede_admitir:
            while pendientes and len(activas) < slots and peticiones[pendientes[0]][0] <= t:
                i = pendientes.pop(0)
                activas[i] = peticiones[i][1]
                inicio[i] = t                      # entra en el lote
        if activas:
            ocupados += len(activas)
            t += 1
            for i in list(activas):
                activas[i] -= 1
                if activas[i] == 0:
                    fin[i] = t
                    del activas[i]
        else:
            t += 1                                 # servidor ocioso esperando llegadas
    ttft_ = [inicio[i] + 1 - peticiones[i][0] for i in inicio]
    lat = [fin[i] - peticiones[i][0] for i in fin]
    return {"pasos totales": t, "tokens/paso": sum(p[1] for p in peticiones) / t,
            "uso de huecos": ocupados / (t * slots), "TTFT medio": np.mean(ttft_),
            "TTFT p95": np.percentile(ttft_, 95), "latencia p95": np.percentile(lat, 95)}

res = {pol: simular(pol, PETICIONES) for pol in ("estatico", "continuo")}
print(f"\n{'':16}{'estático':>12}{'continuo':>12}")
for k in res["estatico"]:
    print(f"{k:16}{res['estatico'][k]:>12.2f}{res['continuo'][k]:>12.2f}")

**Lectura:** con la misma capacidad, el continuo aprovecha mejor los huecos, termina antes y, sobre todo, reduce el **TTFT**: una petición nueva no espera a que acabe un lote ajeno. (Simplificaciones: un paso cuesta lo mismo con cualquier ocupación ≤ 8 y el *prefill* se cuenta como un paso; en un sistema real el prefill compite con el decode y se gestiona aparte.)

## 4 · Cuantización a mano: int8, int4 y calidad

Cuantizar es guardar cada peso con menos bits: `entero = redondear(peso / escala)` y, al usarlo, `peso ≈ entero × escala`. La **escala** se calcula con el valor máximo de cada **bloque** de pesos. La granularidad es la decisión clave: una escala para toda la matriz (*global*), una por canal (fila) o una por *grupos* de 32–128 valores.

In [ ]:
def cuantizar(W, bits, esquema, grupo=32):
    """Cuantización simétrica a mano. Devuelve (enteros, escalas) con escala = max|w| / qmax.
    esquema: 'global' (1 escala), 'canal' (1 por fila) o 'grupo' (1 por cada `grupo` valores de una fila)."""
    qmax = 2 ** (bits - 1) - 1                                  # 127 para int8, 7 para int4
    if esquema == "global":
        escala = W.abs().max().reshape(1, 1)
        base = W
    elif esquema == "canal":
        escala = W.abs().amax(dim=1, keepdim=True)
        base = W
    else:                                                       # grupo
        base = W.reshape(W.shape[0], -1, grupo)
        escala = base.abs().amax(dim=2, keepdim=True)
    escala = escala.clamp(min=1e-8) / qmax
    q = torch.round(base / escala).clamp(-qmax, qmax).to(torch.int8)
    return q, escala

def descuantizar(q, escala, forma):
    return (q.float() * escala).reshape(forma)

def error_relativo(W, W_hat):
    return ((W - W_hat).norm() / W.norm()).item()

def empaquetar_4bits(q):
    """Dos valores de 4 bits en un solo byte: así se ahorra de verdad la memoria."""
    u = (q.reshape(-1) + 8).to(torch.uint8)                     # -7..7 → 1..15
    return (u[0::2] << 4) | u[1::2]

def desempaquetar_4bits(p, forma):
    u = torch.stack([p >> 4, p & 0x0F], dim=1).reshape(-1)
    return (u.to(torch.int8) - 8).reshape(forma)

W = m.blocks[0].mlp[0].weight.detach()                          # una matriz REAL del modelo (256×64)
for bits, esquema in [(8, "global"), (8, "canal"), (4, "canal"), (4, "grupo")]:
    q, e = cuantizar(W, bits, esquema)
    print(f"int{bits} {esquema:7s} → error relativo {error_relativo(W, descuantizar(q, e, W.shape)):.3%}")

q4, e4 = cuantizar(W, 4, "grupo")
p = empaquetar_4bits(q4)
assert torch.equal(desempaquetar_4bits(p, q4.shape), q4)
print(f"\nEmpaquetado: {q4.numel()} valores de 4 bits caben en {p.numel()} bytes (más las escalas en fp16: {e4.numel() * 2} bytes)")

# --- Por qué importa la granularidad: una fila con valores 30 veces mayores (un «canal atípico»)
W_out = W.clone(); W_out[7] *= 30
resto = [i for i in range(W.shape[0]) if i != 7]
print("\nCon un canal atípico (fila 7 ×30), error medio en las DEMÁS filas:")
for esquema in ("global", "canal"):
    q, e = cuantizar(W_out, 8, esquema)
    W_hat = descuantizar(q, e, W.shape)
    print(f"  int8 {esquema:7s}: {error_relativo(W_out[resto], W_hat[resto]):.3%}")

Ahora lo importante: ¿qué pasa con la **calidad** del modelo? Cuantizamos todas las capas lineales y evaluamos el acierto de la tarea y la divergencia (KL) de sus probabilidades frente al modelo original.

In [ ]:
def cuantizar_modelo(modelo, bits, esquema):
    """Copia del modelo con todas las capas Linear cuantizadas y descuantizadas («fake quant»):
    el ordenador sigue calculando en fp32, pero con los valores que tendrías tras cuantizar."""
    copia = copy.deepcopy(modelo)
    bytes_lineales = 0
    for mod in copia.modules():
        if isinstance(mod, nn.Linear):
            q, e = cuantizar(mod.weight.data, bits, esquema)
            mod.weight.data = descuantizar(q, e, mod.weight.shape)
            bytes_lineales += q.numel() * bits / 8 + e.numel() * 2        # enteros + escalas fp16
    return copia, bytes_lineales

bytes_fp32 = sum(p.numel() * 4 for mod in m.modules() if isinstance(mod, nn.Linear) for p in [mod.weight])
filas = [("fp32 (original)", 32, None)] + [(f"int{b} {e}", b, e) for b, e in
        [(8, "global"), (8, "canal"), (4, "global"), (4, "canal"), (4, "grupo"), (3, "grupo"), (2, "canal")]]
print(f"{'esquema':18} {'memoria':>10} {'acierto':>9} {'KL vs fp32':>11}")
for nombre, bits, esquema in filas:
    if esquema is None:
        modelo_q, memoria = m, bytes_fp32
    else:
        modelo_q, memoria = cuantizar_modelo(m, bits, esquema)
    acierto, logits = evaluar(modelo_q)
    kl = F.kl_div(F.log_softmax(logits, -1), F.log_softmax(LOGITS_FP32, -1), log_target=True,
                  reduction="batchmean").item() / logits.shape[1]
    print(f"{nombre:18} {memoria / 1024:>7.1f} KiB {acierto:>9.1%} {kl:>11.5f}")
print("\nLa memoria baja de forma predecible; la CALIDAD hay que medirla (aquí: acierto y KL frente al fp32).")

**Lectura:** int8 es prácticamente gratis en este juguete e int4 aguanta (la KL sube un poco, el acierto no); a 3 bits la divergencia ya se multiplica y a 2 bits **se dispara** (cientos de veces mayor que en int8) aunque el acierto de esta tarea fácil siga intacto: la KL avisa antes que la métrica de tarea. Dos avisos: (1) el error numérico de una matriz no es la calidad del modelo, y un 100 % de acierto en una tarea fácil no prueba que el modelo grande siga igual de bien en tus casos difíciles; (2) este juguete no tiene atípicos importantes, los modelos grandes sí, y ahí la granularidad pesa mucho más. Regla de oro: **evaluar siempre con tu propio conjunto de evaluación (AI4)** antes de desplegar un modelo cuantizado.

## 5 · Memoria de la KV-cache de un modelo de 7–8 B

Por token, la caché guarda K y V de cada capa: `2 × capas × cabezas_KV × head_dim × bytes`. Con **GQA** (*grouped-query attention*) hay menos cabezas KV que de consulta, y la caché se reduce en proporción.

In [ ]:
def kv_bytes_por_token(capas, kv_heads, head_dim, bytes_elem=2):
    return 2 * capas * kv_heads * head_dim * bytes_elem            # 2 = K y V

MODELOS = {   # configuraciones ILUSTRATIVAS tipo Llama de ~8 B: comprueba el config.json del modelo que uses
    "8B con GQA (8 cabezas KV)":  dict(params=8.0e9, capas=32, kv_heads=8,  head_dim=128),
    "7B sin GQA (32 cabezas KV)": dict(params=7.0e9, capas=32, kv_heads=32, head_dim=128),
}
GiB = 1024 ** 3

print(f"{'modelo':28} {'KV/token':>10} {'8k ctx':>9} {'32k ctx':>9} {'128k ctx':>10}   (KV-cache en fp16, POR PETICIÓN)")
for nombre, c in MODELOS.items():
    por_token = kv_bytes_por_token(c["capas"], c["kv_heads"], c["head_dim"])
    print(f"{nombre:28} {por_token / 1024:>7.0f} KiB " + " ".join(
        f"{por_token * ctx / GiB:>8.1f}G" for ctx in (8_192, 32_768, 131_072)))

def peticiones_concurrentes(c, vram_gib, ctx, bits_pesos=16, bytes_kv=2, uso_gpu=0.9):
    pesos = c["params"] * bits_pesos / 8
    libre = vram_gib * GiB * uso_gpu - pesos
    por_peticion = kv_bytes_por_token(c["capas"], c["kv_heads"], c["head_dim"], bytes_kv) * ctx
    return max(0, int(libre // por_peticion)), libre / GiB

print("\nPeticiones simultáneas que caben (contexto 32k, 90 % de la VRAM disponible):")
print(f"{'modelo':28} {'GPU':>7} {'pesos':>7} {'KV':>6} {'cabe':>6}")
c = MODELOS["8B con GQA (8 cabezas KV)"]
for vram in (24, 80):
    for bits_p, bytes_kv, etiqueta in [(16, 2, "fp16/fp16"), (4, 2, "int4/fp16"), (4, 1, "int4/fp8")]:
        n, libre = peticiones_concurrentes(c, vram, 32_768, bits_p, bytes_kv)
        print(f"{'8B GQA':28} {vram:>4} GB {etiqueta:>10}  {n:>4}   (libre para KV: {libre:.1f} GiB)")

**Lectura:** una sola petición de 32k tokens puede ocupar varios GiB *solo de caché*. Con una GPU de 24 GB y un modelo de 8 B en fp16 apenas caben unas pocas peticiones largas; cuantizar los **pesos** libera VRAM, y cuantizar la **caché** (8 bits) o usar **GQA** multiplica la concurrencia. PagedAttention no reduce el total, pero evita desperdiciarlo por fragmentación.

## 6 · Coste por millón de tokens: servir tú frente a una API

La fórmula es sencilla: `coste_hora / (tokens_por_segundo × 3600 × utilización)`. Lo difícil es la **utilización**: una GPU alquilada cuesta lo mismo esté ocupada o no. Los precios de abajo son **supuestos de ejemplo**: sustitúyelos por tarifas vigentes y por tu throughput medido.

In [ ]:
def coste_por_millon(eur_hora, tokens_s, utilizacion):
    """€ por millón de tokens servidos por tu propia GPU. `utilizacion` = fracción del tiempo con carga."""
    return eur_hora / (tokens_s * 3600 * utilizacion) * 1e6

# SUPUESTOS (cámbialos por los tuyos: miden tu GPU y tu carga, no las de esta página):
EUR_HORA_GPU = 2.0          # alquiler de una GPU de 24–80 GB: comprueba tarifas vigentes
TOKENS_S_SERVIDOR = 1500    # tokens/s AGREGADOS con batching alto: mídelo con tu modelo y vLLM
PRECIO_API = {"entrada": 1.0, "salida": 5.0}      # € por millón de tokens: EJEMPLO, pon la tarifa real
PROPORCION_ENTRADA = 0.75   # 3 de cada 4 tokens son de entrada (prompts largos, respuestas cortas)
precio_mezcla = PROPORCION_ENTRADA * PRECIO_API["entrada"] + (1 - PROPORCION_ENTRADA) * PRECIO_API["salida"]

print(f"API (mezcla {PROPORCION_ENTRADA:.0%} entrada): {precio_mezcla:.2f} €/M tokens\n")
print(f"{'utilización':>12} {'€/M tokens (propia)':>20} {'¿gana a la API?':>16}")
for u in (0.05, 0.15, 0.30, 0.60, 0.90):
    c_ = coste_por_millon(EUR_HORA_GPU, TOKENS_S_SERVIDOR, u)
    print(f"{u:>12.0%} {c_:>20.2f} {('sí' if c_ < precio_mezcla else 'no'):>16}")

coste_mes = EUR_HORA_GPU * 24 * 30
tokens_equilibrio = coste_mes / precio_mezcla                 # millones de tokens/mes
capacidad_mes = TOKENS_S_SERVIDOR * 3600 * 24 * 30 / 1e6      # millones de tokens/mes a plena carga
print(f"\nGPU 24/7: {coste_mes:,.0f} €/mes. Empata con la API a {tokens_equilibrio:,.0f} M tokens/mes "
      f"({tokens_equilibrio / capacidad_mes:.0%} de su capacidad de {capacidad_mes:,.0f} M).")
print("Aún no cuentan: ingeniería, guardias, redundancia (≥ 2 réplicas), cola de picos ni el tiempo de montar y mantener todo.")

## 7 · Prueba de carga contra una API local

Ponemos el modelo detrás de una API FastAPI con un número limitado de «huecos» de inferencia (un semáforo) y lanzamos peticiones concurrentes. Usamos `TestClient` para la prueba funcional y `httpx` con transporte ASGI para la carga: **no se abre ningún puerto ni queda ningún proceso en marcha**.

In [ ]:
import httpx
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class Peticion(BaseModel):
    n_tokens: int = Field(16, ge=1, le=64)

def crear_app(slots: int) -> FastAPI:
    """Servicio con `slots` «huecos» de inferencia: solo ese número de peticiones se procesan a la vez."""
    app = FastAPI(title="Servidor de juguete")
    estado = {"sem": None}                          # el semáforo se crea dentro del bucle de eventos

    @app.get("/health")
    async def health():
        return {"estado": "ok"}

    @app.post("/generar")
    async def generar_ep(p: Peticion):
        if estado["sem"] is None:
            estado["sem"] = asyncio.Semaphore(slots)
        t0 = time.perf_counter()
        async with estado["sem"]:                   # cola de espera cuando todos los huecos están ocupados
            espera = time.perf_counter() - t0
            prompt = torch.randint(0, V, (1, 16))
            await asyncio.to_thread(generar, m, prompt, p.n_tokens, True)
        return {"tokens": p.n_tokens, "espera_ms": round(espera * 1000, 1)}
    return app

# 1) Prueba funcional con TestClient (síncrona, sin abrir puertos ni dejar procesos colgados)
with TestClient(crear_app(2)) as cliente:
    print(cliente.get("/health").json(), cliente.post("/generar", json={"n_tokens": 8}).json())
    print("petición inválida →", cliente.post("/generar", json={"n_tokens": 5000}).status_code)

# 2) Prueba de carga con asyncio + httpx directamente contra la app (ASGI), sin servidor real
async def prueba_carga(app, n_peticiones, concurrencia):
    sem = asyncio.Semaphore(concurrencia)
    transporte = httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transporte, base_url="http://prueba") as cli:
        async def una():
            async with sem:
                t0 = time.perf_counter()
                r = await cli.post("/generar", json={"n_tokens": 16})
                return time.perf_counter() - t0, r.status_code
        t0 = time.perf_counter()
        resultados = await asyncio.gather(*[una() for _ in range(n_peticiones)])
        total = time.perf_counter() - t0
    lat = sorted(r[0] for r in resultados)
    errores = sum(1 for r in resultados if r[1] != 200)
    return {"rps": n_peticiones / total, "p50": lat[len(lat) // 2] * 1000,
            "p95": lat[int(len(lat) * 0.95) - 1] * 1000, "errores": errores}

print(f"\n{'concurrencia':>13} {'peticiones/s':>13} {'p50 (ms)':>10} {'p95 (ms)':>10} {'errores':>8}   (servidor con 2 huecos)")
for conc in (1, 2, 8, 32):
    r = await prueba_carga(crear_app(2), 48, conc)
    print(f"{conc:>13} {r['rps']:>13.1f} {r['p50']:>10.0f} {r['p95']:>10.0f} {r['errores']:>8}")

**Lectura:** al subir la concurrencia por encima de los huecos, el throughput **deja de crecer** (aquí incluso baja un poco: los hilos de PyTorch compiten por la misma CPU) y la latencia (p95) crece porque las peticiones esperan en cola. Lección: atender peticiones en paralelo *sin agruparlas en un lote* no multiplica el rendimiento; por eso los servidores de LLM hacen *batching* dentro del propio motor. Esa curva —peticiones/s frente a p95— es lo que se compara entre configuraciones antes de poner un servicio en producción. Mide con tu patrón de tráfico real (longitudes de prompt y de salida), no con una petición repetida.

## 8 · Esqueleto para Colab: vLLM con una GPU

Lo que necesita GPU queda como esqueleto comentado: el resto del notebook no depende de él.

In [ ]:
# ═══════════ ESQUELETO PARA COLAB (GPU T4 gratuita) — no se ejecuta aquí ═══════════
# vLLM cambia de versión con frecuencia: COMPRUEBA la versión instalada y la documentación vigente
# (https://docs.vllm.ai) y que el modelo siga disponible en Hugging Face antes de ejecutar.
# En Colab: Entorno de ejecución → Cambiar tipo → GPU T4.
#
# !pip install -q vllm
# !python -c "import vllm; print(vllm.__version__)"          # anota la versión en tu informe
#
# # 1) Servidor compatible con la API de OpenAI (en segundo plano)
# !nohup vllm serve Qwen/Qwen2.5-0.5B-Instruct --max-model-len 4096 --dtype half \
#       --gpu-memory-utilization 0.85 > vllm.log 2>&1 &
# !sleep 90 && tail -n 5 vllm.log                              # espera a «Application startup complete»
#
# # 2) Carga: N peticiones concurrentes con el mismo patrón asíncrono de la celda anterior
# import asyncio, httpx, time
# async def una(cli):
#     t0 = time.perf_counter()
#     r = await cli.post("http://localhost:8000/v1/chat/completions", json={
#         "model": "Qwen/Qwen2.5-0.5B-Instruct", "max_tokens": 128,
#         "messages": [{"role": "user", "content": "Explica qué es la KV-cache en 3 frases."}]})
#     return r.json()["usage"]["completion_tokens"], time.perf_counter() - t0
# async def carga(n):
#     async with httpx.AsyncClient(timeout=120) as cli:
#         t0 = time.perf_counter(); res = await asyncio.gather(*[una(cli) for _ in range(n)])
#         return sum(r[0] for r in res) / (time.perf_counter() - t0)
# for n in (1, 8, 32): print(n, "peticiones →", round(await carga(n)), "tokens/s totales")
#
# # 3) Para comparar con tu servicio de juguete, anota: tokens/s totales, TTFT y p95 por concurrencia.
# #    Alternativa: la herramienta de benchmark que incluye vLLM (mira `vllm bench --help` en tu versión).
# !pkill -f "vllm serve"                                       # no dejes el servidor encendido

---
## Resumen

- **Batching** sube el throughput del servidor a costa de la latencia de cada petición; el **continuous batching** evita que un lote espere a su petición más larga.
- **KV-cache**: por paso, el coste de atención pasa de O(t²) a O(t) (de O(T³) a O(T²) en total), pero la caché ocupa memoria que crece con contexto × concurrencia.
- **Cuantizar** reduce la memoria de los pesos (4× con int8, ~7× con int4 por grupos), no la de la KV-cache, y hay que **medir la calidad** con tu evaluación (AI4).
- **TTFT** (prefill) y **tokens/s** (decode) son métricas distintas; el **coste por millón de tokens** solo es bajo si la GPU está ocupada.

## Ejercicios (5)
1. **Batching.** Con `paso_decode`, mide lote 1, 4 y 64 y calcula la *eficiencia de batching* = (tokens/s totales con B) / (B × tokens/s con B=1). ¿Cuál es el lote «razonable» antes de que cada usuario pierda más de la mitad de su velocidad?
2. **Cuantización con atípicos en columnas.** Pon un valor ×30 en una *columna* (no una fila) de `W` y compara el error de int8 *por canal* (fila) frente a int8 *por grupo* de 32. ¿Por qué el esquema por fila ya no te salva?
3. **Memoria de KV-cache.** Calcula cuántas peticiones de contexto 16k caben en una GPU de 48 GB con un modelo de 13 B sin GQA (40 capas, 40 cabezas KV, `head_dim` 128) en fp16, y con la KV-cache en 8 bits. ¿Qué palanca pesa más: cuantizar pesos o cuantizar la caché?
4. **Coste.** Con tus propios supuestos (precio de GPU, tokens/s, tarifa de API), halla la utilización mínima a partir de la cual servir tú mismo sale más barato que la API, y los millones de tokens al mes que eso supone.
5. **Prueba de carga.** Repite la prueba con 1, 2 y 4 huecos (`crear_app(slots)`) a concurrencia 16. ¿Cómo cambian peticiones/s y p95? ¿Por qué no mejora indefinidamente?

> Cuando termines, anota en tu informe qué cambiarías en un servicio real: ¿qué métrica pondrías en una alerta (TTFT p95, cola, memoria de KV-cache)?

In [ ]:
# Ejercicio 1 · eficiencia de batching
base = 1 / tiempos_paso[1]
for B in (1, 4, 64):
    total = B / tiempos_paso[B]
    por_usuario = 1 / tiempos_paso[B]
    print(f"B={B:>3}: total {total:>8,.0f} tok/s · por usuario {por_usuario:>6,.0f} tok/s "
          f"({por_usuario / base:.0%} de B=1) · eficiencia de batching {total / (B * base):.0%}")
B_razonable = max((B for B in tiempos_paso if (1 / tiempos_paso[B]) / base >= 0.5), default=1)
print(f"\nMayor lote en el que cada usuario conserva ≥ 50 % de su velocidad: B={B_razonable}")

In [ ]:
# Ejercicio 2 · atípico en una COLUMNA
W_col = W.clone(); W_col[:, 5] *= 30
resto_cols = [j for j in range(W.shape[1]) if j != 5]
for etiqueta, esquema, grupo in [("por canal (fila)", "canal", 32), ("grupo de 32", "grupo", 32),
                                 ("grupo de 16", "grupo", 16), ("grupo de 8", "grupo", 8)]:
    q, e = cuantizar(W_col, 8, esquema, grupo)
    W_hat = descuantizar(q, e, W.shape)
    print(f"int8 {etiqueta:17s}: error en las demás columnas = {error_relativo(W_col[:, resto_cols], W_hat[:, resto_cols]):.3%}")
print("Con el atípico en una columna, TODAS las filas lo contienen: la escala por fila queda dominada por él.")
print("Un grupo solo contagia a sus vecinos: cuanto más pequeño el grupo, menos valores sufren (a costa de más escalas que guardar).")
print("En LLM reales los atípicos están en las activaciones: por eso existen LLM.int8, AWQ o SmoothQuant.")

In [ ]:
# Ejercicio 3 · memoria de KV-cache de un 13B sin GQA
c13 = dict(params=13e9, capas=40, kv_heads=40, head_dim=128)
for bits_p, bytes_kv, etiqueta in [(16, 2, "pesos fp16, KV fp16"), (4, 2, "pesos int4, KV fp16"),
                                   (16, 1, "pesos fp16, KV 8 bits"), (4, 1, "pesos int4, KV 8 bits")]:
    n, libre = peticiones_concurrentes(c13, 48, 16_384, bits_p, bytes_kv)
    print(f"{etiqueta:24s}: caben {n:>2} peticiones de 16k  (libre para KV: {libre:.1f} GiB)")
print("\nCon pesos fp16 el modelo ocupa 26 GB de 48 y cada petición de 16k pesa ~12,5 GiB de caché en fp16.")
print("Aquí cuantizar la CACHÉ a 8 bits (1→3 peticiones) rinde más que cuantizar los pesos a 4 bits (1→2); las dos juntas, 5.")
print("Y con GQA (8 cabezas KV en vez de 40) la caché sería 5× menor: la arquitectura decide la concurrencia.")

In [ ]:
# Ejercicio 4 · utilización de equilibrio
u_min = EUR_HORA_GPU / (TOKENS_S_SERVIDOR * 3600) * 1e6 / precio_mezcla
millones_mes = u_min * TOKENS_S_SERVIDOR * 3600 * 24 * 30 / 1e6
print(f"Con estos supuestos, servir tú empata a utilización {u_min:.1%} → {millones_mes:,.0f} M tokens/mes en una GPU 24/7.")
print("Por debajo, la API es más barata (y sin operar nada); por encima hay que sumar el coste de ingeniería y redundancia.")

In [ ]:
# Ejercicio 5 · huecos del servidor
print(f"{'huecos':>7} {'peticiones/s':>13} {'p95 (ms)':>10}")
for slots in (1, 2, 4):
    r = await prueba_carga(crear_app(slots), 48, 16)
    print(f"{slots:>7} {r['rps']:>13.1f} {r['p95']:>10.0f}")
print("\nMás huecos NO dan más rendimiento: el hardware es un recurso finito y cada petición ya lo usa entero")
print("(en esta CPU, hilos de PyTorch compitiendo; en una GPU, núcleos y memoria). Para ganar de verdad hay que")
print("agrupar peticiones en un mismo lote (batching dentro del motor). Encontrar ese punto es el objetivo de una prueba de carga.")